group AJ

```
# This is formatted as code
```



In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [6]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

In [7]:
class IntermediateBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_sizes):
        super().__init__()

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=k,
                    padding=k // 2,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            )
            for k in kernel_sizes
        ])


        self.fc = nn.Linear(in_channels, len(kernel_sizes))

    def forward(self, x):


        m = x.mean(dim=(2, 3))


        a = self.fc(m)

        a = torch.softmax(a, dim=1)

        branch_outputs = torch.stack([branch(x) for branch in self.branches], dim=1)

        a = a.view(a.size(0), a.size(1), 1, 1, 1)

        x_prime = (a * branch_outputs).sum(dim=1)
        return x_prime


class OutputBlock(nn.Module):
    def __init__(self, in_channels, num_classes=10, hidden_dims=None, dropout=0.3):
        super().__init__()

        if hidden_dims is None:
            hidden_dims = []

        layers = []
        input_dim = in_channels

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(dropout))
            input_dim = hidden_dim

        layers.append(nn.Linear(input_dim, num_classes))
        self.classifier = nn.Sequential(*layers)

    def forward(self, x):

        m = x.mean(dim=(2, 3))
        logits = self.classifier(m)
        return logits


class BasicNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = IntermediateBlock(in_channels=3, out_channels=64, kernel_sizes=[1, 3, 5])
        self.pool1 = nn.MaxPool2d(2)

        self.block2 = IntermediateBlock(in_channels=64, out_channels=128, kernel_sizes=[1, 3, 5])
        self.pool2 = nn.MaxPool2d(2)

        self.block3 = IntermediateBlock(in_channels=128, out_channels=256, kernel_sizes=[1, 3, 5])

        self.output_block = OutputBlock(
            in_channels=256,
            num_classes=10,
            hidden_dims=[128],
            dropout=0.3
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.pool1(x)

        x = self.block2(x)
        x = self.pool2(x)

        x = self.block3(x)

        logits = self.output_block(x)
        return logits

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BasicNet().to(device)

x = torch.randn(8, 3, 32, 32).to(device)
y = model(x)

print(y.shape)

torch.Size([8, 10])


In [9]:

def evaluate(model, loader, criterion, device):
    model.eval()
    total = 0
    correct = 0
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    avg_loss = running_loss / total
    acc = 100.0 * correct / total
    return avg_loss, acc


def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, device, epochs=50):
    batch_losses = []
    train_accs = []
    test_accs = []
    best_test_acc = 0.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        running_correct = 0
        running_total = 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            batch_losses.append(loss.item())

            preds = outputs.argmax(dim=1)
            running_total += labels.size(0)
            running_correct += (preds == labels).sum().item()

        train_acc = 100.0 * running_correct / running_total
        _, test_acc = evaluate(model, test_loader, criterion, device)

        train_accs.append(train_acc)
        test_accs.append(test_acc)

        if scheduler is not None:
            scheduler.step()

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(f"Epoch {epoch+1:02d}/{epochs} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

    return batch_losses, train_accs, test_accs, best_test_acc, best_state

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BasicNet().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

batch_losses, train_accs, test_accs, best_test_acc, best_state = train_model(
    model,
    train_loader,
    test_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs=50
)

print("Best test accuracyis :", best_test_acc)

Epoch 01/50 | Train Acc: 44.72% | Test Acc: 52.80%
Epoch 02/50 | Train Acc: 59.15% | Test Acc: 60.72%
Epoch 03/50 | Train Acc: 64.60% | Test Acc: 55.98%
Epoch 04/50 | Train Acc: 67.92% | Test Acc: 58.83%
Epoch 05/50 | Train Acc: 70.27% | Test Acc: 66.92%
Epoch 06/50 | Train Acc: 72.33% | Test Acc: 61.99%
Epoch 07/50 | Train Acc: 73.80% | Test Acc: 72.16%
Epoch 08/50 | Train Acc: 75.48% | Test Acc: 69.54%
Epoch 09/50 | Train Acc: 76.54% | Test Acc: 73.48%
Epoch 10/50 | Train Acc: 77.84% | Test Acc: 62.56%
Epoch 11/50 | Train Acc: 78.86% | Test Acc: 68.91%
Epoch 12/50 | Train Acc: 79.74% | Test Acc: 73.14%
Epoch 13/50 | Train Acc: 80.44% | Test Acc: 70.51%
Epoch 14/50 | Train Acc: 81.24% | Test Acc: 76.43%
Epoch 15/50 | Train Acc: 81.86% | Test Acc: 73.74%
Epoch 16/50 | Train Acc: 82.48% | Test Acc: 73.91%
Epoch 17/50 | Train Acc: 83.05% | Test Acc: 66.23%
Epoch 18/50 | Train Acc: 83.64% | Test Acc: 76.21%
Epoch 19/50 | Train Acc: 84.31% | Test Acc: 79.36%
Epoch 20/50 | Train Acc: 84.87%